# FASE 1: Preparación y limpieza de datos

In [1]:
import pandas as pd

### Carga de datos

In [2]:
teams=pd.read_csv("../DATA/teams.csv")
stats=pd.read_csv("../DATA/teamStats.csv")

Para ordenar una columna respecto sus valores, usar `.sort_values("nombre de la columna")` 

### 1.1 Unión de datos:
 Se suministrán dos archivos .csv. Deben unir (merge/join) los DataFrames para identificar los equipos asociados a cada uno de los partidos y consolidar la información necesaria en una sola estrucutura


In [3]:
team_stats=pd.merge(teams[['teamId','abbreviation','displayName','slug']],stats,how='right',on='teamId')

In [4]:
team_stats.to_csv("../DATA/team_stats.csv",index=False)

### 1.2 Filtrado
Una vez unificados los datos, deben limpiar el DataFrame y conservar únicamente los registros correspondientes a los 10 equipos más representativos del 2024

In [5]:
equip_rep=team_stats[team_stats['slug']
                     .isin(['esp.barcelona', 
                            'esp.real_madrid', 
                            'ger.dortmund', 
                            'ger.bayern_munich', 
                            'fra.psg', 'eng.arsenal', 
                            'eng.chelsea', 'eng.liverpool', 
                            'eng.man_city', 'esp.atletico_madrid'])]

In [6]:
equip_rep['slug'].value_counts()

slug
eng.chelsea            95
esp.real_madrid        94
eng.liverpool          91
eng.arsenal            91
esp.barcelona          90
ger.bayern_munich      90
fra.psg                89
eng.man_city           89
ger.dortmund           87
esp.atletico_madrid    87
Name: count, dtype: int64

In [7]:
equip_rep.to_csv("../DATA/equip_rep.csv",index=False)

### 1.3 Análisis de valores atípicos

In [8]:
equip_rep.info()

<class 'pandas.core.frame.DataFrame'>
Index: 903 entries, 852 to 102348
Data columns (total 36 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   teamId              903 non-null    int64  
 1   abbreviation        903 non-null    object 
 2   displayName         903 non-null    object 
 3   slug                903 non-null    object 
 4   seasonType          903 non-null    int64  
 5   eventId             903 non-null    int64  
 6   teamOrder           903 non-null    int64  
 7   possessionPct       872 non-null    float64
 8   foulsCommitted      872 non-null    float64
 9   yellowCards         872 non-null    float64
 10  redCards            872 non-null    float64
 11  offsides            872 non-null    float64
 12  wonCorners          872 non-null    float64
 13  saves               872 non-null    float64
 14  totalShots          872 non-null    float64
 15  shotsOnTarget       872 non-null    float64
 16  shotPct 

In [9]:
equip_rep_est=equip_rep.describe(include='number',percentiles=[.1, .25, .5, .75, .9, .95, .99]).T

In [10]:
equip_rep_est.assign(**{'big_min':lambda x:x['25%']-1.5 * (x['75%']-x['25%'])},
                     **{'big_max':lambda x:x['75%']+1.5 * (x['75%']-x['25%'])})

,count,mean,std,min,10%,25%,50%,75%,90%,95%,99%,max,big_min,big_max
teamId,903.0,309.694352,275.409648,83.0,86.00,124.00,359.0,364.00,382.0,1068.00,1068.000,1068.0,-2.360000e+02,724.000
seasonType,903.0,12948.249169,392.943343,12282.0,12654.00,12655.00,12883.0,13446.50,13561.0,13682.00,13682.000,13682.0,1.146775e+04,14633.750
eventId,903.0,723093.677741,18726.875215,691203.0,704418.40,704788.50,720362.0,740621.50,748285.2,757701.90,758194.880,760553.0,6.510390e+05,794371.000
teamOrder,903.0,0.500554,0.500277,0.0,0.00,0.00,1.0,1.00,1.0,1.00,1.000,1.0,-1.500000e+00,2.500
possessionPct,872.0,59.954243,11.684723,0.0,45.51,53.00,61.0,68.40,73.9,76.20,81.358,85.2,2.990000e+01,91.500
foulsCommitted,872.0,10.149083,3.588254,0.0,6.00,8.00,10.0,12.00,15.0,16.00,19.000,23.0,2.000000e+00,18.000
yellowCards,872.0,1.605505,1.326362,0.0,0.00,1.00,1.0,2.00,3.0,4.00,5.000,8.0,-5.000000e-01,3.500
redCards,872.0,0.082569,0.303169,0.0,0.00,0.00,0.0,0.00,0.0,1.00,1.000,3.0,0.000000e+00,0.000
offsides,872.0,1.628440,1.567324,0.0,0.00,0.00,1.0,2.00,4.0,5.00,6.290,12.0,-3.000000e+00,5.000
wonCorners,872.0,6.122706,3.427764,0.0,2.00,4.00,6.0,8.00,10.0,12.00,15.580,26.0,-2.000000e+00,14.000


In [11]:
columns=['possessionPct',
       'foulsCommitted', 'yellowCards', 'redCards', 'offsides', 'wonCorners',
       'saves', 'totalShots', 'shotsOnTarget', 'shotPct', 'penaltyKickGoals',
       'penaltyKickShots', 'accuratePasses', 'totalPasses', 'passPct',
       'accurateCrosses', 'totalCrosses', 'crossPct', 'totalLongBalls',
       'accurateLongBalls', 'longballPct', 'blockedShots', 'effectiveTackles',
       'totalTackles', 'tacklePct', 'interceptions', 'effectiveClearance',
       'totalClearance']

In [12]:
summary_data = []

for col in columns:
    total_records = len(equip_rep[col])
    zero_count = (equip_rep[col] == 0).sum()
    nan_count = equip_rep[col].isna().sum()

    summary_data.append({
        'Column': col,
        'Total_Records': total_records,
        'Zero_Count': zero_count,
        'NaN_Count': nan_count,
        'Zero_or_NaN_Count': zero_count + nan_count
    })

summary_df = pd.DataFrame(summary_data)

In [13]:
summary_df

,Column,Total_Records,Zero_Count,NaN_Count,Zero_or_NaN_Count
0,possessionPct,903,4,31,35
1,foulsCommitted,903,4,31,35
2,yellowCards,903,202,31,233
3,redCards,903,805,31,836
4,offsides,903,227,31,258
5,wonCorners,903,14,31,45
6,saves,903,101,31,132
7,totalShots,903,3,31,34
8,shotsOnTarget,903,8,31,39
9,shotPct,903,8,31,39


In [14]:
equip_rep.query("possessionPct.isna()")["slug"].value_counts()

slug
ger.dortmund           6
eng.arsenal            5
eng.liverpool          5
eng.chelsea            4
ger.bayern_munich      4
esp.barcelona          4
eng.man_city           2
esp.atletico_madrid    1
Name: count, dtype: int64

#### Reemplazo valores que no reflejan la realidad del fenomeno

In [15]:
cols_to_replace = [
    'possessionPct', 'foulsCommitted', 'accuratePasses', 'totalPasses', 'passPct',
    'totalLongBalls', 'accurateLongBalls', 'longballPct', 'effectiveTackles',
    'tacklePct', 'interceptions', 'effectiveClearance', 'totalClearance'
]


equip_rep = equip_rep.astype({col: 'float64' for col in cols_to_replace})


equip_rep = equip_rep.assign(
    **{
        col: equip_rep.groupby('slug')[col].transform(
            lambda x: x.where(x != 0, x.median())
        )
        for col in cols_to_replace
    }
)

In [16]:
summary_data_outliers = []

for col in columns:
    total_records = len(equip_rep[col])
    zero_count = (equip_rep[col] == 0).sum()
    nan_count = equip_rep[col].isna().sum()

    summary_data_outliers.append({
        'Column': col,
        'Total_Records': total_records,
        'Zero_Count': zero_count,
        'NaN_Count': nan_count,
        'Zero_or_NaN_Count': zero_count + nan_count
    })

summary_df_outliers = pd.DataFrame(summary_data_outliers)


In [17]:
summary_df_outliers

,Column,Total_Records,Zero_Count,NaN_Count,Zero_or_NaN_Count
0,possessionPct,903,0,31,31
1,foulsCommitted,903,0,31,31
2,yellowCards,903,202,31,233
3,redCards,903,805,31,836
4,offsides,903,227,31,258
5,wonCorners,903,14,31,45
6,saves,903,101,31,132
7,totalShots,903,3,31,34
8,shotsOnTarget,903,8,31,39
9,shotPct,903,8,31,39


In [18]:
equip_rep_clean=equip_rep.dropna()

In [19]:
summary_data_clean = []

for col in columns:
    total_records = len(equip_rep[col])
    zero_count = (equip_rep_clean[col] == 0).sum()
    nan_count = equip_rep_clean[col].isna().sum()

    summary_data_clean.append({
        'Column': col,
        'Total_Records': total_records,
        'Zero_Count': zero_count,
        'NaN_Count': nan_count,
        'Zero_or_NaN_Count': zero_count + nan_count
    })

summary_df_clean = pd.DataFrame(summary_data_clean)

summary_df_clean

,Column,Total_Records,Zero_Count,NaN_Count,Zero_or_NaN_Count
0,possessionPct,903,0,0,0
1,foulsCommitted,903,0,0,0
2,yellowCards,903,202,0,202
3,redCards,903,805,0,805
4,offsides,903,227,0,227
5,wonCorners,903,14,0,14
6,saves,903,101,0,101
7,totalShots,903,3,0,3
8,shotsOnTarget,903,8,0,8
9,shotPct,903,8,0,8


In [20]:
equip_rep_clean["slug"].value_counts()

slug
esp.real_madrid        94
eng.chelsea            91
fra.psg                89
eng.man_city           87
eng.liverpool          86
ger.bayern_munich      86
esp.barcelona          86
eng.arsenal            86
esp.atletico_madrid    86
ger.dortmund           81
Name: count, dtype: int64

In [21]:
equip_rep_clean.shape

(872, 36)

In [22]:
equip_rep_clean.to_csv("../DATA/equip_rep_clean.csv",index=False)

### DESICION FINAL. FASE 1

1. Se encontró un total de 31 juegos sin registro alguno, los mismo fueron eliminados.
2. Se encontraron valores nulos en variables específicas que, siguiendo la naturaleza de un partido de futbol, las mismas deberían contar con al menos un valor numérico mayor a cero; bajo esta premisa, se realizó la imputación de la mediana por equipo a cada variable,logrando así, suavizar este registro sin afectar el estilo de juego de cada equipo.
3. Se encontraron valores átipicos por encima de algunos bigotes, se decidió mantenerlos puesto que aunque son encesarios que salen un poco de lo regular, son posibles y dicen mucho sobre el estilo de juego y nivel de cada equipo. 
Estas acciones dieron paso a un DataFrame limpio y listo para aplicar la Fase 2.